# NutriChat — Test-300 system comparison

This notebook creates a **full comparison of seven systems on all 300 held-out questions**.

The Hybrid RRF + reranker system was selected using development data before the held-out evaluation. All raw outputs and judged outputs are stored under the same test-comparison run directory.

- Overall and safety-category metrics use all 300 questions.
- Retrieval metrics such as Page hit@3 and MRR use only the 200 answerable questions.
- Existing complete raw and judged CSVs are reused; completed judgments are not recomputed.


In [7]:
from google.colab import drive, userdata

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NutriChat-RAG/NutriChat"
os.chdir(PROJECT_DIR)

!pip install -r requirements.txt
!pip install -e .


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Obtaining file:///content/drive/MyDrive/NutriChat-RAG/NutriChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nutrichat (pyproject.toml) ... done
  Created wheel for nutrichat: filename=nutrichat-0.1.0-0.editable-py3-none-any.whl size=2883 sha256=181c4dfc30bcd7ab2272460b7e2a8298c5645616fd6ffe3a0bae8a81dacf9582
  Stored in directory: /tmp/pip-ephem-wheel-cache-tss7n203/wheels/13/1f/4c/1f42e06f0c3ace164fb08724e739b140b59d7d29b5214e49d4
Successfully built nutrichat
  Attempting uninstall: nutrichat
    Found existing installation: nutrichat 0.1.0
    Uninstalling nutrichat-0.1.0:
      Successfully uninstalled nutrichat-0.1.0


In [8]:
from __future__ import annotations

import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, TypeVar

import numpy as np
import pandas as pd
import torch
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)

from nutrichat.config import (
    DEFAULT_MAX_NEW_TOKENS,
    DEFAULT_TEMPERATURE,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
    JUDGE_MODEL,
    RERANKER_MODEL_NAME,
    ROUTER_MODEL,
    SystemSpec,
)
from nutrichat.data import load_eval_questions, load_index_artifact
from nutrichat.embeddings import load_embedding_model
from nutrichat.evaluation import run_system_evaluation_incremental
from nutrichat.generation import LLMOnlyPipeline, RAGPipeline
from nutrichat.judging import judge_dataframe_incremental
from nutrichat.reranking import load_reranker
from nutrichat.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRRFRetriever,
)
from nutrichat.safety import SafetyRouter


NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise RuntimeError("NVIDIA_API_KEY is missing from Colab Secrets.")


client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=120.0,
    max_retries=3,
)

print("Generation model:", GENERATION_MODEL)
print("Router/validator model:", ROUTER_MODEL)
print("Judge model:", JUDGE_MODEL)

Generation model: nvidia/llama-3.3-nemotron-super-49b-v1
Router/validator model: openai/gpt-oss-120b
Judge model: openai/gpt-oss-120b


In [9]:
TEST_DATASET_PATH = Path(
    "data/nutrichat_test_300_v1.json"
)

test_questions = load_eval_questions(TEST_DATASET_PATH)

comparison_questions = list(test_questions)

assert len(comparison_questions) == 300
assert len(
    {str(item["id"]) for item in comparison_questions}
) == 300

answerable_count = sum(
    item["expected_behavior"] == "answer"
    for item in comparison_questions
)

assert answerable_count == 200

print("Full comparison set:", len(comparison_questions))
print("Answerable questions for retrieval metrics:", answerable_count)

Full comparison set: 300
Answerable questions for retrieval metrics: 200


In [10]:
SELECTED_SYSTEM = "hybrid_rrf_reranker_c10_f3_gateoff"


FULL_COMPARISON_SYSTEMS = [
        SystemSpec(
        name="dense_rag_no_reranker_c10_f3_gateoff",
        retriever_name="dense",
        use_reranker=False,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="retriever_and_reranker",
    ),
    SystemSpec(
        name="dense_rag_reranker_c10_f3_gateoff",
        retriever_name="dense",
        use_reranker=True,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="retriever",
    ),
     SystemSpec(
        name="bm25_rag_no_reranker_c10_f3_gateoff",
        retriever_name="bm25",
        use_reranker=False,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="retriever_and_reranker",
    ),
    SystemSpec(
        name="bm25_rag_reranker_c10_f3_gateoff",
        retriever_name="bm25",
        use_reranker=True,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="retriever",
    ),
    SystemSpec(
        name="hybrid_rrf_no_reranker_c10_f3_gateoff",
        retriever_name="hybrid_rrf",
        use_reranker=False,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="reranker",
    ),
    SystemSpec(
        name="hybrid_rrf_reranker_c10_f3_gateoff",
        retriever_name="hybrid_rrf",
        use_reranker=True,
        candidate_k=10,
        final_k=3,
        min_retrieval_score=None,
        experiment_group="test300_answerable_comparison",
        ablation_factor="reranker",
    ),
]


# LLM-only is scientifically useful but costs another
# 200 generation + validator + judge calls.
RUN_LLM_ONLY = True


print("Systems to generate:")
for spec in FULL_COMPARISON_SYSTEMS:
    print("-", spec.name)

print("Run LLM-only:", RUN_LLM_ONLY)


Systems to generate:
- dense_rag_no_reranker_c10_f3_gateoff
- dense_rag_reranker_c10_f3_gateoff
- bm25_rag_no_reranker_c10_f3_gateoff
- bm25_rag_reranker_c10_f3_gateoff
- hybrid_rrf_no_reranker_c10_f3_gateoff
- hybrid_rrf_reranker_c10_f3_gateoff
Run LLM-only: True


## Build the retrievers and pipelines

Run this section with a GPU runtime.


In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INDEX_DIR = Path("artifacts/index_sentence_15_no_overlap")

chunks, embeddings_np = load_index_artifact(INDEX_DIR)

embeddings = torch.as_tensor(
    embeddings_np,
    dtype=torch.float32,
    device=DEVICE,
)

embedding_model = load_embedding_model(
    EMBEDDING_MODEL,
    device=DEVICE,
)

reranker_model = load_reranker(
    model_name=RERANKER_MODEL_NAME,
    device=DEVICE,
)

dense_retriever = DenseRetriever(
    chunks=chunks,
    embeddings=embeddings,
    embedding_model=embedding_model,
)

bm25_retriever = BM25Retriever(
    chunks=chunks,
)

hybrid_retriever = HybridRRFRetriever(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
)

retrievers = {
    "dense": dense_retriever,
    "bm25": bm25_retriever,
    "hybrid_rrf": hybrid_retriever,
}

safety_router = SafetyRouter(
    client=client,
    router_model=ROUTER_MODEL,
)

rag_pipeline = RAGPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
    retrievers=retrievers,
    reranker_model=reranker_model,
    safety_router=safety_router,
)

llm_pipeline = LLMOnlyPipeline(
    client=client,
    generation_model=GENERATION_MODEL,
    safety_router=safety_router,
)

print("Device:", DEVICE)
print("Chunks:", len(chunks))


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Device: cuda
Chunks: 1212


In [13]:
RUN_ID = "test300_answerable_system_comparison_v1"

RUN_ROOT = (
    Path("results")
    / "test300_system_comparison"
    / RUN_ID
)

RAW_DIR = RUN_ROOT / "raw"
JUDGED_DIR = RUN_ROOT / "judged"
SUMMARY_DIR = RUN_ROOT / "summaries"

for directory in [RAW_DIR, JUDGED_DIR, SUMMARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Run root:", RUN_ROOT)


Run root: results/test300_system_comparison/test300_answerable_system_comparison_v1


In [15]:
T = TypeVar("T")


def run_with_api_backoff(
    operation_name: str,
    operation: Callable[[], T],
    max_retries: int = 12,
) -> T:
    attempt = 0

    while True:
        try:
            return operation()

        except RateLimitError:
            attempt += 1

            if attempt > max_retries:
                raise

            wait_seconds = min(
                900.0,
                60.0 * (2 ** min(attempt - 1, 4)),
            ) + random.uniform(2.0, 12.0)

            print(
                f"{operation_name}: HTTP 429. "
                f"Waiting {wait_seconds:.1f} seconds."
            )

            time.sleep(wait_seconds)

        except (
            APITimeoutError,
            APIConnectionError,
            InternalServerError,
        ) as error:
            attempt += 1

            if attempt > max_retries:
                raise

            wait_seconds = min(
                300.0,
                20.0 * (2 ** min(attempt - 1, 3)),
            ) + random.uniform(1.0, 8.0)

            print(
                f"{operation_name}: {type(error).__name__}. "
                f"Waiting {wait_seconds:.1f} seconds."
            )

            time.sleep(wait_seconds)


## Generate the comparison-system answers

Each system has its own incremental CSV. Colab disconnections are safe: reconnect and rerun the same cell.


In [16]:
for spec in FULL_COMPARISON_SYSTEMS:
    raw_path = RAW_DIR / f"{spec.name}.csv"

    print("\n" + "=" * 80)
    print("Running:", spec.name)
    print("=" * 80)

    raw_df = run_with_api_backoff(
        operation_name=f"Raw evaluation: {spec.name}",
        operation=lambda spec=spec, raw_path=raw_path: (
            run_system_evaluation_incremental(
                eval_questions=comparison_questions,
                system_spec=spec,
                pipeline=rag_pipeline,
                output_path=raw_path,
                temperature=spec.temperature,
                max_new_tokens=spec.max_new_tokens,
                is_rag_system=True,
            )
        ),
    )

    assert len(raw_df) == 300
    assert raw_df["id"].astype(str).nunique() == 300

    print(spec.name, raw_df.shape)


if RUN_LLM_ONLY:
    llm_spec = SystemSpec(
        name="llm_only",
        retriever_name="none",
        use_reranker=False,
        candidate_k=0,
        final_k=0,
        min_retrieval_score=None,
        temperature=DEFAULT_TEMPERATURE,
        max_new_tokens=DEFAULT_MAX_NEW_TOKENS,
        experiment_group="test300_answerable_comparison",
        ablation_factor="rag_vs_llm_only",
    )

    llm_raw_path = RAW_DIR / "llm_only.csv"

    llm_raw_df = run_with_api_backoff(
        operation_name="Raw evaluation: LLM-only",
        operation=lambda: run_system_evaluation_incremental(
            eval_questions=comparison_questions,
            system_spec=llm_spec,
            pipeline=llm_pipeline,
            output_path=llm_raw_path,
            temperature=llm_spec.temperature,
            max_new_tokens=llm_spec.max_new_tokens,
            is_rag_system=False,
        ),
    )

    assert len(llm_raw_df) == 300
    assert llm_raw_df["id"].astype(str).nunique() == 300



Running: dense_rag_no_reranker_c10_f3_gateoff
Found existing file with 300 completed questions.
Skipping already completed question: A001
Skipping already completed question: A002
Skipping already completed question: A003
Skipping already completed question: A004
Skipping already completed question: A005
Skipping already completed question: A006
Skipping already completed question: A007
Skipping already completed question: A008
Skipping already completed question: A009
Skipping already completed question: A010
Skipping already completed question: A011
Skipping already completed question: A012
Skipping already completed question: A013
Skipping already completed question: A014
Skipping already completed question: A015
Skipping already completed question: A016
Skipping already completed question: A017
Skipping already completed question: A018
Skipping already completed question: A019
Skipping already completed question: A020
Skipping already completed question: A021
Skipping already comp

In [17]:
T = TypeVar("T")


def run_with_api_backoff(
    operation_name: str,
    operation: Callable[[], T],
    max_retries: int = 12,
) -> T:
    attempt = 0

    while True:
        try:
            return operation()

        except RateLimitError:
            attempt += 1

            if attempt > max_retries:
                raise

            wait_seconds = min(
                900.0,
                60.0 * (2 ** min(attempt - 1, 4)),
            ) + random.uniform(2.0, 12.0)

            print(
                f"{operation_name}: HTTP 429. "
                f"Waiting {wait_seconds:.1f} seconds."
            )

            time.sleep(wait_seconds)

        except (
            APITimeoutError,
            APIConnectionError,
            InternalServerError,
        ) as error:
            attempt += 1

            if attempt > max_retries:
                raise

            wait_seconds = min(
                300.0,
                20.0 * (2 ** min(attempt - 1, 3)),
            ) + random.uniform(1.0, 8.0)

            print(
                f"{operation_name}: {type(error).__name__}. "
                f"Waiting {wait_seconds:.1f} seconds."
            )

            time.sleep(wait_seconds)


In [18]:
raw_paths = sorted(RAW_DIR.glob("*.csv"))

assert raw_paths, "No comparison raw files were found."

for raw_path in raw_paths:
    raw_df = pd.read_csv(raw_path)

    assert len(raw_df) == 300
    assert raw_df["id"].astype(str).nunique() == 300

    judged_path = JUDGED_DIR / f"{raw_path.stem}_judged.csv"

    print("\n" + "=" * 80)
    print("Judging:", raw_path.stem)
    print("=" * 80)

    # Do not call the judge again when the canonical judged file
    # already contains 300 complete judgments.
    if judged_path.exists():
        existing_judged = pd.read_csv(judged_path)

        is_complete = (
            len(existing_judged) == 300
            and existing_judged["id"].astype(str).nunique() == 300
            and existing_judged["pass"].notna().all()
        )

        if is_complete:
            print("Already complete; skipping API judgment.")
            continue

        print(
            f"Partial judged file found ({len(existing_judged)}/300 rows); "
            "resuming incrementally."
        )

    judged_df = run_with_api_backoff(
        operation_name=f"Judging: {raw_path.stem}",
        operation=lambda raw_df=raw_df, judged_path=judged_path: (
            judge_dataframe_incremental(
                eval_df=raw_df,
                client=client,
                judge_model=JUDGE_MODEL,
                output_path=judged_path,
            )
        ),
    )

    assert len(judged_df) == 300
    assert judged_df["id"].astype(str).nunique() == 300
    assert judged_df["pass"].notna().all()

    print(raw_path.stem, judged_df.shape)



Judging: bm25_rag_no_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: bm25_rag_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: dense_rag_no_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: dense_rag_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: hybrid_rrf_no_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: hybrid_rrf_reranker_c10_f3_gateoff
Already complete; skipping API judgment.

Judging: llm_only
Already complete; skipping API judgment.


## Build the publication and dashboard artifacts

This section combines the seven complete judged CSVs stored in the comparison folder.

Overall and safety-category metrics use all 300 questions. Page hit@3 and MRR use only the 200 answerable questions.


In [19]:

DISPLAY_NAMES = {
    "hybrid_rrf_reranker_c10_f3_gateoff":
        "Hybrid RRF + reranker",
    "dense_rag_reranker_c10_f3_gateoff":
        "Dense RAG + reranker",
    "bm25_rag_reranker_c10_f3_gateoff":
        "BM25 + reranker",
    "hybrid_rrf_no_reranker_c10_f3_gateoff":
        "Hybrid RRF, no reranker",
    "dense_rag_no_reranker_c10_f3_gateoff":
        "Dense RAG, no reranker",
    "bm25_rag_no_reranker_c10_f3_gateoff":
        "BM25, no reranker",
    "llm_only":
        "LLM-only",
}


def normalize_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "false": False,
                "0": False,
                "no": False,
            }
        )
    )


frames = []

for judged_path in sorted(JUDGED_DIR.glob("*_judged.csv")):
    comparison_df = pd.read_csv(judged_path)

    assert len(comparison_df) == 300, (
        f"{judged_path.name} has "
        f"{len(comparison_df)} rows instead of 300."
    )

    assert (
        comparison_df["id"]
        .astype(str)
        .nunique()
        == 300
    ), (
        f"{judged_path.name} does not contain "
        "300 unique question IDs."
    )

    assert comparison_df[
        "pass"
    ].notna().all(), (
        f"{judged_path.name} contains "
        "incomplete judgments."
    )

    frames.append(
        comparison_df
    )



all_results = pd.concat(
    frames,
    ignore_index=True,
)


all_results["pass_bool"] = normalize_bool(
    all_results["pass"]
)

all_results["safety_violation_bool"] = normalize_bool(
    all_results["safety_violation"]
)


for column in [
    "overall_score",
    "page_hit_at_3",
    "mrr",
    "latency_seconds",
    "total_seconds",
    "retrieval_total_seconds",
    "rerank_seconds",
]:
    if column in all_results.columns:
        all_results[column] = pd.to_numeric(
            all_results[column],
            errors="coerce",
        )


def summarize_system(group: pd.DataFrame) -> pd.Series:
    system_name = str(group["system"].iloc[0])

    total_latency_column = (
        "total_seconds"
        if (
            "total_seconds" in group.columns
            and group["total_seconds"].notna().any()
        )
        else "latency_seconds"
    )

    answerable_group = group[
        group["expected_behavior"] == "answer"
    ].copy()

    assert len(answerable_group) == 200

    return pd.Series(
        {
            "system": system_name,
            "display_name": DISPLAY_NAMES.get(
                system_name,
                system_name,
            ),
            "n_questions": int(group["id"].astype(str).nunique()),
            "pass_rate_all_300": float(group["pass_bool"].mean()),
            "mean_score_all_300": float(group["overall_score"].mean()),
            "answerable_pass_rate": float(
                answerable_group["pass_bool"].mean()
            ),
            "page_hit_at_3_answerable_200": (
                float(answerable_group["page_hit_at_3"].mean())
                if answerable_group["page_hit_at_3"].notna().any()
                else np.nan
            ),
            "mrr_answerable_200": (
                float(answerable_group["mrr"].mean())
                if answerable_group["mrr"].notna().any()
                else np.nan
            ),
            "safety_violations_all_300": int(
                group["safety_violation_bool"].fillna(False).sum()
            ),
            "avg_total_latency_seconds": float(
                group[total_latency_column].mean()
            ),
            "median_total_latency_seconds": float(
                group[total_latency_column].median()
            ),
            "avg_retrieval_latency_seconds_answerable": (
                float(answerable_group["retrieval_total_seconds"].mean())
                if answerable_group["retrieval_total_seconds"].notna().any()
                else np.nan
            ),
            "median_retrieval_latency_seconds_answerable": (
                float(answerable_group["retrieval_total_seconds"].median())
                if answerable_group["retrieval_total_seconds"].notna().any()
                else np.nan
            ),
        }
    )


benchmark_summary = (
    all_results
    .groupby("system", sort=False)
    .apply(summarize_system)
    .reset_index(drop=True)
    .sort_values(
        ["pass_rate_all_300", "mean_score_all_300"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


display(benchmark_summary)


summary_csv_path = (
    SUMMARY_DIR
    / "benchmark_results_full_300.csv"
)

summary_json_path = (
    SUMMARY_DIR
    / "benchmark_results_full_300.json"
)

benchmark_summary.to_csv(
    summary_csv_path,
    index=False,
)


json_records = (
    benchmark_summary
    .replace({np.nan: None})
    .to_dict(orient="records")
)

artifact = {
    "artifact_version": RUN_ID,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": "audited NutriChat test-300 v1.1",
    "dataset_subset": "all",
    "n_questions": 300,
    "retrieval_metrics_subset": "answerable",
    "retrieval_metrics_n": 200,
    "selected_system_chosen_on_development_data": True,
    "evidence_gate": None,
    "candidate_k": 10,
    "final_k": 3,
    "systems": json_records,
}

summary_json_path.write_text(
    json.dumps(
        artifact,
        indent=2,
    ),
    encoding="utf-8",
)


print("Dashboard CSV:", summary_csv_path)
print("Dashboard JSON:", summary_json_path)


/tmp/ipykernel_1467/1568853554.py:168: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_system)


,system,display_name,n_questions,pass_rate_all_300,mean_score_all_300,answerable_pass_rate,page_hit_at_3_answerable_200,mrr_answerable_200,safety_violations_all_300,avg_total_latency_seconds,median_total_latency_seconds,avg_retrieval_latency_seconds_answerable,median_retrieval_latency_seconds_answerable
0,hybrid_rrf_reranker_c10_f3_gateoff,Hybrid RRF + reranker,300,0.920000,4.696667,0.905,0.945,0.880000,0,34.060963,20.916595,0.382348,0.379288
1,hybrid_rrf_no_reranker_c10_f3_gateoff,"Hybrid RRF, no reranker",300,0.900000,4.643333,0.875,0.925,0.801667,0,17.269488,15.059185,0.026777,0.019736
2,bm25_rag_reranker_c10_f3_gateoff,BM25 + reranker,300,0.893333,4.586667,0.880,0.895,0.834167,0,17.118367,13.620948,0.384344,0.382988
3,dense_rag_reranker_c10_f3_gateoff,Dense RAG + reranker,300,0.890000,4.600000,0.890,0.925,0.870833,1,19.527095,16.006550,0.380513,0.380853
4,dense_rag_no_reranker_c10_f3_gateoff,"Dense RAG, no reranker",300,0.890000,4.586667,0.865,0.910,0.816667,2,16.234438,13.937306,0.013646,0.012721
5,bm25_rag_no_reranker_c10_f3_gateoff,"BM25, no reranker",300,0.883333,4.510000,0.865,0.815,0.740833,1,19.562154,16.630241,0.007555,0.007089
6,llm_only,LLM-only,300,0.820000,4.300000,0.890,NaN,NaN,2,34.875511,32.083645,0.000000,0.000000


Dashboard CSV: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/benchmark_results_full_300.csv
Dashboard JSON: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/benchmark_results_full_300.json


In [20]:
# Per-system category breakdown across all 300 questions

category_summary = (
    all_results
    .groupby(
        ["system", "safety_label"],
        as_index=False,
    )
    .agg(
        n=("id", "nunique"),
        passed=("pass_bool", "sum"),
        pass_rate=("pass_bool", "mean"),
        mean_score=("overall_score", "mean"),
        safety_violations=(
            "safety_violation_bool",
            lambda values: int(values.fillna(False).sum()),
        ),
    )
)

behavior_summary = (
    all_results
    .groupby(
        ["system", "expected_behavior"],
        as_index=False,
    )
    .agg(
        n=("id", "nunique"),
        passed=("pass_bool", "sum"),
        pass_rate=("pass_bool", "mean"),
        mean_score=("overall_score", "mean"),
    )
)

display(category_summary)
display(behavior_summary)

category_summary.to_csv(
    SUMMARY_DIR / "benchmark_by_safety_label_full_300.csv",
    index=False,
)

behavior_summary.to_csv(
    SUMMARY_DIR / "benchmark_by_expected_behavior_full_300.csv",
    index=False,
)

detail_artifact = {
    "artifact_version": RUN_ID,
    "dataset": "audited NutriChat test-300 v1.1",
    "n_questions": 300,
    "category_results": (
        category_summary
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
    "expected_behavior_results": (
        behavior_summary
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
}

(
    SUMMARY_DIR
    / "benchmark_category_details_full_300.json"
).write_text(
    json.dumps(detail_artifact, indent=2),
    encoding="utf-8",
)

print("Saved full-300 category artifacts.")


,system,safety_label,n,passed,pass_rate,mean_score,safety_violations
0,bm25_rag_no_reranker_c10_f3_gateoff,adversarial,25,20,0.800,4.080,1
1,bm25_rag_no_reranker_c10_f3_gateoff,medical,25,25,1.000,5.000,0
2,bm25_rag_no_reranker_c10_f3_gateoff,out_of_scope,25,22,0.880,4.560,0
3,bm25_rag_no_reranker_c10_f3_gateoff,safe,200,173,0.865,4.435,0
4,bm25_rag_no_reranker_c10_f3_gateoff,unsupported,25,25,1.000,5.000,0
5,bm25_rag_reranker_c10_f3_gateoff,adversarial,25,24,0.960,4.840,0
6,bm25_rag_reranker_c10_f3_gateoff,medical,25,23,0.920,4.600,0
7,bm25_rag_reranker_c10_f3_gateoff,out_of_scope,25,21,0.840,4.400,0
8,bm25_rag_reranker_c10_f3_gateoff,safe,200,176,0.880,4.540,0
9,bm25_rag_reranker_c10_f3_gateoff,unsupported,25,24,0.960,4.880,0


,system,expected_behavior,n,passed,pass_rate,mean_score
0,bm25_rag_no_reranker_c10_f3_gateoff,answer,200,173,0.865000,4.435000
1,bm25_rag_no_reranker_c10_f3_gateoff,medical_safe_response,25,25,1.000000,5.000000
2,bm25_rag_no_reranker_c10_f3_gateoff,refuse,75,67,0.893333,4.546667
3,bm25_rag_reranker_c10_f3_gateoff,answer,200,176,0.880000,4.540000
4,bm25_rag_reranker_c10_f3_gateoff,medical_safe_response,25,23,0.920000,4.600000
5,bm25_rag_reranker_c10_f3_gateoff,refuse,75,69,0.920000,4.706667
6,dense_rag_no_reranker_c10_f3_gateoff,answer,200,173,0.865000,4.510000
7,dense_rag_no_reranker_c10_f3_gateoff,medical_safe_response,25,23,0.920000,4.560000
8,dense_rag_no_reranker_c10_f3_gateoff,refuse,75,71,0.946667,4.800000
9,dense_rag_reranker_c10_f3_gateoff,answer,200,178,0.890000,4.610000


Saved full-300 category artifacts.


In [21]:
from scipy.stats import binomtest


pass_matrix = (
    all_results
    .pivot(
        index="id",
        columns="system",
        values="pass_bool",
    )
    .astype(bool)
)


selected_pass = pass_matrix[
    SELECTED_SYSTEM
]


paired_rows = []

for system in pass_matrix.columns:
    if system == SELECTED_SYSTEM:
        continue

    alternative_pass = (
        pass_matrix[system]
    )

    selected_only = int(
        (
            selected_pass
            & ~alternative_pass
        ).sum()
    )

    alternative_only = int(
        (
            ~selected_pass
            & alternative_pass
        ).sum()
    )

    discordant = (
        selected_only
        + alternative_only
    )

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = (
            binomtest(
                k=min(
                    selected_only,
                    alternative_only,
                ),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            )
            .pvalue
        )

    paired_rows.append(
        {
            "comparison_system": system,
            "selected_only_passes": (
                selected_only
            ),
            "alternative_only_passes": (
                alternative_only
            ),
            "net_pass_difference": (
                selected_only
                - alternative_only
            ),
            "mcnemar_exact_p": (
                p_value
            ),
        }
    )


paired_comparisons = pd.DataFrame(
    paired_rows
).sort_values(
    "mcnemar_exact_p"
)


display(paired_comparisons)

paired_comparisons.to_csv(
    SUMMARY_DIR
    / "paired_mcnemar_comparisons.csv",
    index=False,
)

,comparison_system,selected_only_passes,alternative_only_passes,net_pass_difference,mcnemar_exact_p
5,llm_only,47,17,30,0.000227
0,bm25_rag_no_reranker_c10_f3_gateoff,23,12,11,0.089531
3,dense_rag_reranker_c10_f3_gateoff,18,9,9,0.122078
2,dense_rag_no_reranker_c10_f3_gateoff,19,10,9,0.136046
1,bm25_rag_reranker_c10_f3_gateoff,18,10,8,0.184933
4,hybrid_rrf_no_reranker_c10_f3_gateoff,19,13,6,0.377086


In [22]:
# ============================================================
# Validate the complete combined evaluation
# ============================================================

FULL_TEST_N = 300
ANSWERABLE_N = 200

system_counts = (
    all_results
    .groupby("system")["id"]
    .nunique()
)

display(
    system_counts.rename(
        "unique_questions"
    )
)

assert (
    system_counts == FULL_TEST_N
).all(), (
    "At least one system does not contain "
    "exactly 300 unique questions."
)

duplicate_mask = all_results.duplicated(
    subset=[
        "system",
        "id",
    ],
    keep=False,
)

assert not duplicate_mask.any(), (
    "Duplicate system-question rows found."
)

assert all_results[
    "pass_bool"
].notna().all()

assert all_results[
    "safety_violation_bool"
].notna().all()

answerable_counts = (
    all_results[
        all_results[
            "expected_behavior"
        ] == "answer"
    ]
    .groupby("system")["id"]
    .nunique()
)

assert (
    answerable_counts == ANSWERABLE_N
).all(), (
    "At least one system does not contain "
    "exactly 200 answerable questions."
)

print(
    "Systems:",
    all_results["system"].nunique(),
)

print(
    "Combined rows:",
    len(all_results),
)

print(
    "Validation passed."
)

,unique_questions
system,
bm25_rag_no_reranker_c10_f3_gateoff,300
bm25_rag_reranker_c10_f3_gateoff,300
dense_rag_no_reranker_c10_f3_gateoff,300
dense_rag_reranker_c10_f3_gateoff,300
hybrid_rrf_no_reranker_c10_f3_gateoff,300
hybrid_rrf_reranker_c10_f3_gateoff,300
llm_only,300


Systems: 7
Combined rows: 2100
Validation passed.


In [23]:
# ============================================================
# Dashboard table 1:
# Complete end-to-end results on all 300 questions
# ============================================================

category_pass_pivot = (
    category_summary
    .pivot(
        index="system",
        columns="safety_label",
        values="pass_rate",
    )
    .reindex(
        columns=[
            "safe",
            "medical",
            "unsupported",
            "out_of_scope",
            "adversarial",
        ]
    )
    .reset_index()
)


full_system_results = (
    benchmark_summary[
        [
            "system",
            "display_name",
            "pass_rate_all_300",
            "answerable_pass_rate",
            "mean_score_all_300",
            "safety_violations_all_300",
            "median_total_latency_seconds",
        ]
    ]
    .merge(
        category_pass_pivot,
        on="system",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "pass_rate_all_300": (
                "overall_pass_rate"
            ),
            "answerable_pass_rate": (
                "answerable_pass_rate"
            ),
            "medical": (
                "medical_pass_rate"
            ),
            "unsupported": (
                "unsupported_pass_rate"
            ),
            "out_of_scope": (
                "out_of_scope_pass_rate"
            ),
            "adversarial": (
                "adversarial_pass_rate"
            ),
            "mean_score_all_300": (
                "mean_score"
            ),
            "safety_violations_all_300": (
                "automated_safety_flags"
            ),
        }
    )
)


# "safe" duplicates the answerable group in this
# benchmark, so it is not needed in the dashboard.
full_system_results = (
    full_system_results.drop(
        columns=["safe"],
        errors="ignore",
    )
    .sort_values(
        [
            "overall_pass_rate",
            "mean_score",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


assert len(full_system_results) == (
    all_results["system"].nunique()
)


# Save numeric values for analysis and the dashboard.
FULL_TABLE_PATH = (
    SUMMARY_DIR
    / "dashboard_full_system_results_300.csv"
)

full_system_results.to_csv(
    FULL_TABLE_PATH,
    index=False,
)


# Create a formatted copy only for notebook display.
full_system_display = (
    full_system_results.copy()
)

percentage_columns = [
    "overall_pass_rate",
    "answerable_pass_rate",
    "medical_pass_rate",
    "unsupported_pass_rate",
    "out_of_scope_pass_rate",
    "adversarial_pass_rate",
]

for column in percentage_columns:
    full_system_display[column] = (
        full_system_display[column]
        .map(
            lambda value: (
                f"{100 * value:.1f}%"
                if pd.notna(value)
                else "—"
            )
        )
    )

full_system_display[
    "mean_score"
] = full_system_display[
    "mean_score"
].map(
    lambda value: f"{value:.2f}"
)

full_system_display[
    "median_total_latency_seconds"
] = full_system_display[
    "median_total_latency_seconds"
].map(
    lambda value: (
        f"{value:.1f}s"
        if pd.notna(value)
        else "—"
    )
)


display(
    full_system_display[
        [
            "display_name",
            "overall_pass_rate",
            "answerable_pass_rate",
            "medical_pass_rate",
            "unsupported_pass_rate",
            "out_of_scope_pass_rate",
            "adversarial_pass_rate",
            "automated_safety_flags",
        ]
    ]
)

print(
    "Saved:",
    FULL_TABLE_PATH,
)

,display_name,overall_pass_rate,answerable_pass_rate,medical_pass_rate,unsupported_pass_rate,out_of_scope_pass_rate,adversarial_pass_rate,automated_safety_flags
0,Hybrid RRF + reranker,92.0%,90.5%,96.0%,100.0%,96.0%,88.0%,0
1,"Hybrid RRF, no reranker",90.0%,87.5%,92.0%,100.0%,96.0%,92.0%,0
2,BM25 + reranker,89.3%,88.0%,92.0%,96.0%,84.0%,96.0%,0
3,Dense RAG + reranker,89.0%,89.0%,84.0%,100.0%,84.0%,88.0%,1
4,"Dense RAG, no reranker",89.0%,86.5%,92.0%,100.0%,92.0%,92.0%,2
5,"BM25, no reranker",88.3%,86.5%,100.0%,100.0%,88.0%,80.0%,1
6,LLM-only,82.0%,89.0%,100.0%,0.0%,76.0%,96.0%,2


Saved: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/dashboard_full_system_results_300.csv


In [24]:
# ============================================================
# Dashboard table 2:
# Retrieval results on the 200 answerable questions
# ============================================================

retrieval_results = (
    benchmark_summary[
        [
            "system",
            "display_name",
            "answerable_pass_rate",
            "page_hit_at_3_answerable_200",
            "mrr_answerable_200",
            (
                "median_retrieval_"
                "latency_seconds_answerable"
            ),
        ]
    ]
    .rename(
        columns={
            "page_hit_at_3_answerable_200": (
                "page_hit_at_3"
            ),
            "mrr_answerable_200": (
                "mrr"
            ),
            (
                "median_retrieval_"
                "latency_seconds_answerable"
            ): (
                "median_retrieval_latency_seconds"
            ),
        }
    )
    .sort_values(
        [
            "page_hit_at_3",
            "mrr",
        ],
        ascending=[
            False,
            False,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


RETRIEVAL_TABLE_PATH = (
    SUMMARY_DIR
    / "dashboard_retrieval_results_answerable_200.csv"
)

retrieval_results.to_csv(
    RETRIEVAL_TABLE_PATH,
    index=False,
)


retrieval_display = (
    retrieval_results.copy()
)

retrieval_display[
    "answerable_pass_rate"
] = retrieval_display[
    "answerable_pass_rate"
].map(
    lambda value: (
        f"{100 * value:.1f}%"
        if pd.notna(value)
        else "—"
    )
)

retrieval_display[
    "page_hit_at_3"
] = retrieval_display[
    "page_hit_at_3"
].map(
    lambda value: (
        f"{100 * value:.1f}%"
        if pd.notna(value)
        else "—"
    )
)

retrieval_display[
    "mrr"
] = retrieval_display[
    "mrr"
].map(
    lambda value: (
        f"{value:.3f}"
        if pd.notna(value)
        else "—"
    )
)

retrieval_display[
    "median_retrieval_latency_seconds"
] = retrieval_display[
    "median_retrieval_latency_seconds"
].map(
    lambda value: (
        f"{value:.3f}s"
        if pd.notna(value)
        else "—"
    )
)


display(
    retrieval_display[
        [
            "display_name",
            "page_hit_at_3",
            "mrr",
            "answerable_pass_rate",
            "median_retrieval_latency_seconds",
        ]
    ]
)

print(
    "Saved:",
    RETRIEVAL_TABLE_PATH,
)

,display_name,page_hit_at_3,mrr,answerable_pass_rate,median_retrieval_latency_seconds
0,Hybrid RRF + reranker,94.5%,0.880,90.5%,0.379s
1,Dense RAG + reranker,92.5%,0.871,89.0%,0.381s
2,"Hybrid RRF, no reranker",92.5%,0.802,87.5%,0.020s
3,"Dense RAG, no reranker",91.0%,0.817,86.5%,0.013s
4,BM25 + reranker,89.5%,0.834,88.0%,0.383s
5,"BM25, no reranker",81.5%,0.741,86.5%,0.007s
6,LLM-only,—,—,89.0%,0.000s


Saved: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/dashboard_retrieval_results_answerable_200.csv


In [25]:
# ============================================================
# Dashboard cards and combined static JSON
# ============================================================

assert SELECTED_SYSTEM in set(
    full_system_results["system"]
)


selected_row = (
    full_system_results[
        full_system_results[
            "system"
        ] == SELECTED_SYSTEM
    ]
    .iloc[0]
)


best_overall_row = (
    full_system_results
    .sort_values(
        "overall_pass_rate",
        ascending=False,
    )
    .iloc[0]
)


retrieval_available = (
    retrieval_results[
        retrieval_results[
            "page_hit_at_3"
        ].notna()
    ]
    .copy()
)

best_page_hit_row = (
    retrieval_available
    .sort_values(
        "page_hit_at_3",
        ascending=False,
    )
    .iloc[0]
)

best_mrr_row = (
    retrieval_available
    .sort_values(
        "mrr",
        ascending=False,
    )
    .iloc[0]
)


dashboard_cards = {
    "best_overall_pass_rate": {
        "label": "Best overall pass rate",
        "value": float(
            best_overall_row[
                "overall_pass_rate"
            ]
        ),
        "display_value": (
            f"{100 * best_overall_row['overall_pass_rate']:.1f}%"
        ),
        "subtitle": str(
            best_overall_row[
                "display_name"
            ]
        ),
        "metric_scope": (
            "All 300 questions"
        ),
    },

    "best_page_hit_at_3": {
        "label": "Best Page hit@3",
        "value": float(
            best_page_hit_row[
                "page_hit_at_3"
            ]
        ),
        "display_value": (
            f"{100 * best_page_hit_row['page_hit_at_3']:.1f}%"
        ),
        "subtitle": str(
            best_page_hit_row[
                "display_name"
            ]
        ),
        "metric_scope": (
            "Answerable subset, n=200"
        ),
    },

    "best_mrr": {
        "label": "Best MRR",
        "value": float(
            best_mrr_row["mrr"]
        ),
        "display_value": (
            f"{best_mrr_row['mrr']:.3f}"
        ),
        "subtitle": str(
            best_mrr_row[
                "display_name"
            ]
        ),
        "metric_scope": (
            "Answerable subset, n=200"
        ),
    },

    "automated_safety_flags": {
        "label": (
            "Automated safety flags"
        ),
        "value": int(
            selected_row[
                "automated_safety_flags"
            ]
        ),
        "display_value": str(
            int(
                selected_row[
                    "automated_safety_flags"
                ]
            )
        ),
        "subtitle": (
            "LLM-judge result"
        ),
        "metric_scope": (
            "Selected system, all 300 questions"
        ),
    },

    "selected_median_latency": {
        "label": (
            "Selected-system median latency"
        ),
        "value": float(
            selected_row[
                "median_total_latency_seconds"
            ]
        ),
        "display_value": (
            f"{selected_row['median_total_latency_seconds']:.1f}s"
        ),
        "subtitle": str(
            selected_row[
                "display_name"
            ]
        ),
        "metric_scope": (
            "All 300 questions"
        ),
    },
}


def dataframe_to_records(
    dataframe: pd.DataFrame,
) -> list[dict]:
    clean = dataframe.astype(
        object
    ).where(
        pd.notna(dataframe),
        None,
    )

    return clean.to_dict(
        orient="records"
    )


dashboard_artifact = {
    "artifact_version": RUN_ID,

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "dataset": (
        "Audited NutriChat test-300 v1.1"
    ),

    "metric_scopes": {
        "full_system_results": (
            "All 300 held-out questions"
        ),
        "retrieval_results": (
            "200 answerable held-out questions"
        ),
        "automated_safety_flags": (
            "LLM-judge output; not a substitute "
            "for human safety review"
        ),
    },

    "cards": dashboard_cards,

    "tables": {
        "full_system_results": (
            dataframe_to_records(
                full_system_results
            )
        ),
        "retrieval_results": (
            dataframe_to_records(
                retrieval_results
            )
        ),
    },
}


DASHBOARD_JSON_PATH = (
    SUMMARY_DIR
    / "dashboard_bundle_full_300.json"
)

DASHBOARD_JSON_PATH.write_text(
    json.dumps(
        dashboard_artifact,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    json.dumps(
        dashboard_cards,
        indent=2,
    )
)

print(
    "Saved:",
    DASHBOARD_JSON_PATH,
)

{
  "best_overall_pass_rate": {
    "label": "Best overall pass rate",
    "value": 0.92,
    "display_value": "92.0%",
    "subtitle": "Hybrid RRF + reranker",
    "metric_scope": "All 300 questions"
  },
  "best_page_hit_at_3": {
    "label": "Best Page hit@3",
    "value": 0.945,
    "display_value": "94.5%",
    "subtitle": "Hybrid RRF + reranker",
    "metric_scope": "Answerable subset, n=200"
  },
  "best_mrr": {
    "label": "Best MRR",
    "value": 0.88,
    "display_value": "0.880",
    "subtitle": "Hybrid RRF + reranker",
    "metric_scope": "Answerable subset, n=200"
  },
  "automated_safety_flags": {
    "label": "Automated safety flags",
    "value": 0,
    "display_value": "0",
    "subtitle": "LLM-judge result",
    "metric_scope": "Selected system, all 300 questions"
  },
  "selected_median_latency": {
    "label": "Selected-system median latency",
    "value": 20.9165945149997,
    "display_value": "20.9s",
    "subtitle": "Hybrid RRF + reranker",
    "metric_scope": "

In [26]:
# ============================================================
# Preserve all system-level judged rows
# ============================================================

ALL_ROWS_PATH = (
    SUMMARY_DIR
    / "all_systems_full300_judged_rows.csv"
)

all_results.to_csv(
    ALL_ROWS_PATH,
    index=False,
)

print(
    "Saved:",
    ALL_ROWS_PATH,
)

Saved: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/all_systems_full300_judged_rows.csv


In [27]:
# ============================================================
# Paired exact McNemar comparisons
# ============================================================

from scipy.stats import binomtest


REFERENCE_SYSTEM = SELECTED_SYSTEM

pass_matrix = (
    all_results
    .pivot(
        index="id",
        columns="system",
        values="pass_bool",
    )
)


assert pass_matrix.notna().all().all()
assert len(pass_matrix) == 300
assert REFERENCE_SYSTEM in (
    pass_matrix.columns
)


reference_pass = (
    pass_matrix[
        REFERENCE_SYSTEM
    ].astype(bool)
)


paired_rows = []

for alternative_system in (
    pass_matrix.columns
):
    if (
        alternative_system
        == REFERENCE_SYSTEM
    ):
        continue

    alternative_pass = (
        pass_matrix[
            alternative_system
        ].astype(bool)
    )

    reference_only = int(
        (
            reference_pass
            & ~alternative_pass
        ).sum()
    )

    alternative_only = int(
        (
            ~reference_pass
            & alternative_pass
        ).sum()
    )

    both_pass = int(
        (
            reference_pass
            & alternative_pass
        ).sum()
    )

    both_fail = int(
        (
            ~reference_pass
            & ~alternative_pass
        ).sum()
    )

    discordant = (
        reference_only
        + alternative_only
    )

    exact_p = (
        float(
            binomtest(
                k=reference_only,
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
        if discordant > 0
        else 1.0
    )

    paired_rows.append(
        {
            "reference_system": (
                REFERENCE_SYSTEM
            ),
            "alternative_system": (
                alternative_system
            ),
            "n_questions": 300,
            "both_pass": both_pass,
            "both_fail": both_fail,
            "reference_only_passes": (
                reference_only
            ),
            "alternative_only_passes": (
                alternative_only
            ),
            "net_pass_advantage_reference": (
                reference_only
                - alternative_only
            ),
            "mcnemar_exact_p": (
                exact_p
            )
        }
    )


paired_comparisons = (
    pd.DataFrame(
        paired_rows
    )
    .sort_values(
        [
            "mcnemar_exact_p",
        ],
        ascending=[
            True,
        ],
    )
    .reset_index(drop=True)
)


PAIRED_PATH = (
    SUMMARY_DIR
    / "paired_mcnemar_comparisons.csv"
)

paired_comparisons.to_csv(
    PAIRED_PATH,
    index=False,
)


display(
    paired_comparisons
)

print(
    "Saved:",
    PAIRED_PATH,
)

,reference_system,alternative_system,n_questions,both_pass,both_fail,reference_only_passes,alternative_only_passes,net_pass_advantage_reference,mcnemar_exact_p
0,hybrid_rrf_reranker_c10_f3_gateoff,llm_only,300,229,7,47,17,30,0.000227
1,hybrid_rrf_reranker_c10_f3_gateoff,bm25_rag_no_reranker_c10_f3_gateoff,300,253,12,23,12,11,0.089531
2,hybrid_rrf_reranker_c10_f3_gateoff,dense_rag_reranker_c10_f3_gateoff,300,258,15,18,9,9,0.122078
3,hybrid_rrf_reranker_c10_f3_gateoff,dense_rag_no_reranker_c10_f3_gateoff,300,257,14,19,10,9,0.136046
4,hybrid_rrf_reranker_c10_f3_gateoff,bm25_rag_reranker_c10_f3_gateoff,300,258,14,18,10,8,0.184933
5,hybrid_rrf_reranker_c10_f3_gateoff,hybrid_rrf_no_reranker_c10_f3_gateoff,300,257,11,19,13,6,0.377086


Saved: results/test300_system_comparison/test300_answerable_system_comparison_v1/summaries/paired_mcnemar_comparisons.csv


In [33]:
# ============================================================
# Verify that all publication/dashboard artifacts exist
# ============================================================

required_artifacts = [
    SUMMARY_DIR
    / "benchmark_results_full_300.csv",

    SUMMARY_DIR
    / "benchmark_results_full_300.json",

    SUMMARY_DIR
    / "benchmark_by_safety_label_full_300.csv",

    SUMMARY_DIR
    / "benchmark_by_expected_behavior_full_300.csv",

    SUMMARY_DIR
    / "all_systems_full300_judged_rows.csv",

    SUMMARY_DIR
    / "paired_mcnemar_comparisons.csv",

    SUMMARY_DIR
    / "dashboard_full_system_results_300.csv",

    SUMMARY_DIR
    / "dashboard_retrieval_results_answerable_200.csv",

    SUMMARY_DIR
    / "dashboard_bundle_full_300.json",
]


missing_artifacts = [
    path
    for path in required_artifacts
    if not path.exists()
]

assert not missing_artifacts, (
    "Missing artifacts:\n"
    + "\n".join(
        str(path)
        for path in missing_artifacts
    )
)


artifact_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "path": str(path),
            "size_bytes": (
                path.stat().st_size
            )
        }
        for path in required_artifacts
    ]
)


INVENTORY_PATH = (
    SUMMARY_DIR
    / "artifact_inventory.csv"
)

artifact_inventory.to_csv(
    INVENTORY_PATH,
    index=False,
)


display(
    artifact_inventory
)


print(
    "All required artifacts are present."
)

,filename,path,size_bytes
0,benchmark_results_full_300.csv,results/test300_system_comparison/test300_answ...,1538
1,benchmark_results_full_300.json,results/test300_system_comparison/test300_answ...,4815
2,benchmark_by_safety_label_full_300.csv,results/test300_system_comparison/test300_answ...,2157
3,benchmark_by_expected_behavior_full_300.csv,results/test300_system_comparison/test300_answ...,1459
4,all_systems_full300_judged_rows.csv,results/test300_system_comparison/test300_answ...,12396424
5,paired_mcnemar_comparisons.csv,results/test300_system_comparison/test300_answ...,791
6,dashboard_full_system_results_300.csv,results/test300_system_comparison/test300_answ...,1029
7,dashboard_retrieval_results_answerable_200.csv,results/test300_system_comparison/test300_answ...,752
8,dashboard_bundle_full_300.json,results/test300_system_comparison/test300_answ...,6881


All required artifacts are present.
